# FraudShield — Colab master runner

Every GPU-bound step of this project, in one notebook: **voice generation**,
**document-consistency evaluation**, **video-KYC generation + evaluation**, and
**GNN training**. Each writes its real results straight back into Supabase and
back into the dataset bundles, so nothing has to be pasted by hand afterwards.

## Why this replaces the old per-task notebooks

`docs/SESSION_HANDOFF.md` says not to clone the repo, because `data/generated/`
is gitignored and a clone would arrive empty — so earlier notebooks embedded
backend source files as `repr()`-encoded strings and needed data zipped up
through `files.upload()`. That was the right call at the time and it caused the
two documented Windows-path bugs.

`backend/tools/storage_sync.py` removed the reason for it: the dataset now lives
in Supabase Storage. So this notebook just **clones the repo** (real code, always
current) and **pulls the data** (real data, always current). No embedded source
strings, no manual zip transfer, no path-separator translation.

## Run order

Cells 1–3 are setup and are needed by everything. After that each task section is
independent — run only the ones you need. **Runtime → Change runtime type → GPU**
before starting.

## 1 · Clone the repo

In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/officialayush23/red_hat_vs_blue_hat_attack.git"
REPO_DIR = pathlib.Path("/content/red_hat_vs_blue_hat_attack")

# If the repo is private, put a GitHub personal access token here and the URL
# becomes https://<token>@github.com/... . Leave empty for a public repo.
GITHUB_TOKEN = ""

url = REPO_URL if not GITHUB_TOKEN else REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", url, str(REPO_DIR)], check=True)

BACKEND = REPO_DIR / "backend"
os.chdir(BACKEND)
sys.path.insert(0, str(BACKEND))
print("repo at", REPO_DIR)
print("HEAD:", subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"],
                              capture_output=True, text=True).stdout.strip())

## 2 · Credentials

Pasted at runtime, never committed. `SUPABASE_SERVICE_ROLE_KEY` is the write key
— it bypasses RLS, so treat this notebook as sensitive while it holds one.

Copy the values from your local `backend/.env`.

In [ ]:
from getpass import getpass

os.environ["SUPABASE_URL"] = input("SUPABASE_URL: ").strip()
os.environ["SUPABASE_ANON_KEY"] = getpass("SUPABASE_ANON_KEY: ").strip()
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = getpass("SUPABASE_SERVICE_ROLE_KEY: ").strip()
hf = getpass("HF_TOKEN (blank if the models you need are public): ").strip()
if hf:
    os.environ["HF_TOKEN"] = hf

# Written to backend/.env because db/supabase_client.py loads it by path.
(BACKEND / ".env").write_text("\n".join(
    f"{k}={os.environ[k]}" for k in
    ["SUPABASE_URL", "SUPABASE_ANON_KEY", "SUPABASE_SERVICE_ROLE_KEY"] + (["HF_TOKEN"] if hf else [])
))
print("credentials set")

## 3 · Pull the dataset from Supabase Storage

Only the bundles the task you are about to run needs. Each bundle is
sha256-verified before extraction, so a truncated download can never quietly
become a partial dataset that every metric is then computed over.

In [ ]:
!pip install -q supabase python-dotenv

# Edit to match the task you are running. Full list:
#   attacks, synthetic_customers,
#   voice_attacks, voice_bonafide,
#   document_attacks, document_bonafide,
#   phishing_attacks, phishing_bonafide,
#   video_kyc_attacks, video_kyc_bonafide, video_kyc_reference
BUNDLES = "document_attacks,document_bonafide,voice_attacks,voice_bonafide,synthetic_customers"

!python tools/storage_sync.py pull --only {BUNDLES}
!python tools/storage_sync.py status

## 3b · Make evaluation checkpoints survive the runtime  *(strongly recommended)*

Scoring is deterministic — a detector sees a file path and nothing else — so a
score computed once is valid forever. The eval scripts now checkpoint every
image to `backend/.eval_cache/` and resume from it.

But that directory lives on the Colab VM, which is deleted when the runtime
ends. On 2026-09-01 a usage limit killed a run at image **290 of 680** and every
minute of it was lost.

Mounting Drive puts the checkpoint somewhere that outlives the runtime, so a
killed run resumes at 290 instead of restarting at 1. Skip this cell only if you
do not want to connect Drive — everything still works, it just cannot resume
across a runtime death.

In [ ]:
from google.colab import drive
import pathlib, os

drive.mount("/content/drive")

PERSISTENT = pathlib.Path("/content/drive/MyDrive/fraudshield_eval_cache")
PERSISTENT.mkdir(parents=True, exist_ok=True)

local = pathlib.Path("/content/red_hat_vs_blue_hat_attack/backend/.eval_cache")
if local.is_symlink() or local.exists():
    if not local.is_symlink():
        # Move anything already computed in this runtime into Drive first,
        # rather than discarding it to make room for the symlink.
        for f in local.glob("*.json"):
            target = PERSISTENT / f.name
            if not target.exists():
                target.write_bytes(f.read_bytes())
        import shutil; shutil.rmtree(local)
    else:
        local.unlink()
local.symlink_to(PERSISTENT, target_is_directory=True)

print("checkpoints ->", PERSISTENT)
for f in sorted(PERSISTENT.glob("*.json")):
    import json as _json
    print(f"  {f.name}: {len(_json.loads(f.read_text()))} cached scores")

---
## A · Document OCR — rapidocr on GPU, 680 cases

### The bake-off is over. Don't re-run it.

An earlier version of this section ran three engines over the same images because
I had no measurement on *these* invoices — clean, known-font, axis-aligned Pillow
renders that no public OCR leaderboard is a proxy for. That measurement now exists,
on the full 680:

| entry | recall | precision | FPR | n |
|---|---|---|---|---|
| **rapidocr** | **1.0000** | 0.9108 | 0.2350 | 680 |
| paddlevl (incumbent) | 0.9125 | 0.8795 | 0.2500 | 120 |

rapidocr won and the question is settled, so this section now runs **rapidocr
only**. Re-scoring tesseract and easyocr would spend a third of a Colab session
each re-deriving a conclusion already in `metrics.json` — and a Colab runtime
already died at image 290/680 once for exactly that reason. If you ever do want
the losers re-measured, `DOC_OCR_BACKEND=tesseract` still works; it is a choice
now, not the default path.

### That 0.2350 FPR was a bug, and it is fixed

23.5% of *legitimate* invoices were being flagged. Cause: `score()` is
mismatched-over-comparable across four fields, so it can only return
{0, 0.25, 0.5, 0.75, 1.0} — and the threshold sits at 0.25. **One character
misread in one of four fields** therefore scored exactly 0.25 and tripped the
flag. `O`→`0` in a bank account is an OCR slip, not fraud. `_values_match()` now
folds the confusable glyph classes on ID-like fields and requires a 0.90 similarity
elsewhere, and a QR that won't decode abstains at 0.5 instead of asserting 1.0.
Verified on 9 cases covering both OCR slips and real substitutions.

**So this run is not a repeat — it is the first measurement of the fixed detector.**
Expect recall to hold at 1.0 and FPR to drop well below 0.2350. If recall falls, the
tolerance is too loose and that is the finding.

### GPU

rapidocr is ONNX graphs through onnxruntime, so GPU is purely which execution
provider is registered — identical weights, identical thresholds, identical
outputs, just faster. A1 installs `onnxruntime-gpu` and A2 sets
`DOC_OCR_USE_GPU=1`, which **raises rather than falling back**, so a run that
quietly reverted to CPU can never be reported as a GPU run.

> **Set the runtime to a GPU first:** Runtime → Change runtime type → T4 GPU.


In [ ]:
# A1 — rapidocr on GPU.
#
# WHY onnxruntime-gpu AND NOT plain onnxruntime: `pip install
# rapidocr-onnxruntime` pulls the CPU wheel, and RapidOCR defaults every
# stage to CPU no matter what is installed -- you have to ask for CUDA. The
# 680-image run that died at image 290 was CPU-bound for exactly this reason.
#
# The two onnxruntime wheels CONFLICT (both provide the `onnxruntime`
# module), so the CPU one is removed first rather than installed over.
#
# tesseract and easyocr are deliberately NOT installed. The bake-off is
# decided; see the cell above.
#
# NO numpy pin, deliberately. Colab's runtime is numpy 2.x (jax, cupy, shap,
# opencv-contrib 4.14 all require >=2). Pinning "numpy<2" here -- a
# constraint belonging to the LOCAL `red` venv where facenet-pytorch lives --
# caused a POISONED KERNEL: pip swapped numpy on disk while the imported
# numpy stayed live in memory, and the next C extension to load hit
#   ValueError: numpy.dtype size changed ... Expected 96 from C header, got 88
# which names numpy but means "restart this kernel". The guard below catches
# that state directly instead of letting it surface as a cryptic dtype error.
!pip uninstall -q -y onnxruntime > /dev/null 2>&1
!pip install -q --upgrade "numpy>=2" rapidocr-onnxruntime onnxruntime-gpu pillow qrcode

import numpy
from importlib.metadata import version as _pkg_version

_in_memory, _on_disk = numpy.__version__, _pkg_version("numpy")
if _in_memory != _on_disk:
    raise SystemExit(
        f"STOP -- restart the runtime.\n\n"
        f"numpy in memory: {_in_memory}\nnumpy on disk:   {_on_disk}\n\n"
        "pip replaced numpy underneath a kernel that had already imported it. "
        "Nothing installed after this point will load reliably.\n\n"
        "Runtime > Restart session, then run cells 1, 2, 3 and A1 again. No need "
        "to re-clone or re-pull; the repo and the pulled data survive a restart."
    )

import onnxruntime, cv2
providers = onnxruntime.get_available_providers()
print("numpy:", _in_memory, "| opencv:", cv2.__version__)
print("onnxruntime:", onnxruntime.__version__, "| providers:", providers)

if "CUDAExecutionProvider" not in providers:
    raise SystemExit(
        "STOP -- no CUDAExecutionProvider.\n\n"
        "Runtime > Change runtime type > T4 GPU, then re-run cells 1, 2, 3 and A1.\n"
        "(If the runtime IS a GPU one, restart the session -- onnxruntime-gpu was "
        "installed after onnxruntime had already been imported.)"
    )

# The real precondition beyond OCR: cv2 must be able to build a QR detector,
# since that does the QR half of the printed-vs-encoded cross-check.
cv2.QRCodeDetector()
print("QR detector OK")

import rapidocr_onnxruntime  # noqa: F401
print("rapidocr + CUDA ready")


In [ ]:
# A2 — score the 680 documents with rapidocr on GPU.
#
# THREE things this cell learned the hard way on 2026-09-01:
#
#  1. Backfill FIRST. evaluation_results.case_id is a foreign key into
#     attack_cases. If the scored cases are not in that table, every insert
#     batch is rejected -- and the eval's persistence block catches it as
#     "non-fatal", so metrics.json fills with real numbers while Supabase
#     receives nothing. Exactly what happened: 480 scored, 0 persisted.
#     Media families need backfill_phase2_artifacts, not just
#     backfill_attack_cases -- that omission was the whole bug.
#  2. capture_output=False was not enough -- subprocess stderr never reached
#     the Colab cell, which is why the failure above left no trace. stderr is
#     merged into stdout and streamed line by line here.
#  3. Interrupted runs RESUME. Every image's score is checkpointed to
#     backend/.eval_cache/ as it is computed (Drive-backed if you ran cell
#     3b), so re-running after a runtime death picks up where it stopped
#     rather than rescoring from image 1. A cached run replays in seconds.
#
# DOC_OCR_USE_GPU=1 raises if CUDA is unavailable rather than falling back,
# so this can never silently become a slow CPU run reported as a fast one.
import os, subprocess, time

def run(cmd, env=None):
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

print("=" * 70, "\nbackfilling attack_cases so the FK can be satisfied\n" + "=" * 70)
rc = run(["python", "generate/run_all_generation.py",
          "--only", "backfill_attack_cases,backfill_phase2_artifacts"])
if rc != 0:
    raise SystemExit(
        f"backfill exited {rc}. Do NOT continue -- without the attack_cases rows "
        "every evaluation_results insert will be rejected by the FK and the run "
        "will look successful while persisting nothing."
    )

print("\n" + "=" * 70, "\nrapidocr (GPU)\n" + "=" * 70)
t0 = time.time()
rc = run(["python", "evaluation/eval_document_consistency.py"],
         env={**os.environ, "DOC_OCR_BACKEND": "rapidocr", "DOC_OCR_USE_GPU": "1"})
print(f"\nexit {rc} in {time.time() - t0:.0f}s")


In [ ]:
# A3 — the fix, measured against the pre-fix baseline.
import json, pathlib
m = json.loads(pathlib.Path("defend/models/metrics.json").read_text())
rows = {k: v for k, v in m.items() if k.startswith("document_consistency_detector")}
print(f"{'entry':52} {'recall':>8} {'prec':>8} {'FPR':>8} {'n':>6}")
for k, v in sorted(rows.items()):
    e = v.get("metrics", v)
    print(f"{k:52} {e.get('recall', 0):8.4f} {e.get('precision', 0):8.4f} "
          f"{e.get('false_positive_rate', 0):8.4f} {e.get('n_samples', 0):6}")

print("""
RESULT (2026-09-01, rapidocr, n=680):

  paddlevl incumbent          recall 0.9125  prec 0.8795  FPR 0.2500  n=120
  rapidocr, before the fix    recall 1.0000  prec 0.9108  FPR 0.2350  n=680
  rapidocr, after the fix     recall 1.0000  prec 0.9375  FPR 0.1600  n=680

FPR 0.2350 -> 0.1600 with recall held at 1.0000: 32 false positives instead of
47, and not one tampered invoice missed. The cause was that score() is
mismatched-over-comparable across four fields, so it can only return
{0, 0.25, 0.5, 0.75, 1.0} while the threshold sits at 0.25 -- a SINGLE
character misread in ONE field scored exactly 0.25 and tripped the flag.
_values_match() now folds confusable glyphs on ID-like fields and requires 0.90
similarity elsewhere; an undecodable QR abstains at 0.5 rather than asserting
1.0.

16% is still high for production. Report it as measured, not as solved.
""")


In [ ]:
# A4 — did it actually reach Supabase? metrics.json updating is NOT proof.
# The whole point of this section is closing the "480 cases, 0 scored" gap,
# and that gap lives in the database, not in a local JSON file.
import os
from supabase import create_client
sb = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_ROLE_KEY"])

for label, pattern in [("document_fraud (attacks)", "document_fraud%"),
                       ("document_bonafide (legit)", "document_bonafide%")]:
    cases = sb.table("attack_cases").select("id", count="exact").like("id", pattern).execute().count
    scored = sb.table("evaluation_results").select("id", count="exact").like("case_id", pattern).execute().count
    print(f"{label:28} attack_cases={cases:>6}   evaluation_results={scored:>6}")

print("\nBoth numbers must be non-zero. If evaluation_results is 0 while attack_cases "
      "is not, re-read the A2 output for the '!! SUPABASE PERSISTENCE FAILED' banner.")

---
## B · Voice — generation, then a spoof-model bake-off

Two jobs. Generation is blocked locally by the Windows pagefile error
(`os error 1455`); the bake-off is new.

### The incumbent and its challengers

`garystafford/wav2vec2-deepfake-voice-detector` (Apache 2.0) currently scores
recall 0.8171, precision 0.8701, **FPR 12.5%** on n=166.

`VOICE_MODEL_ID` now selects the checkpoint, and `eval_voice_spoof.py` records the
model id in the metrics key — so challengers are measured on the identical cases
and cannot overwrite the incumbent.

Two Hugging Face challengers load through the same
`AutoModelForAudioClassification` path and cost one line each to try.

**On AASIST**, which is what the literature actually points at: the AASIST family
(graph-attention over wav2vec2 features) is the stronger published architecture
for ASVspoof-style anti-spoofing, and `lab260/AASIST3` exists. It is **not**
`AutoModelForAudioClassification`-compatible — it needs its own loader — so it is
a separate piece of work, not a `VOICE_MODEL_ID` swap. Worth doing if voice is
the modality you want to lead with; not worth faking compatibility for.

Chatterbox pins `torch==2.6.0`, so **run section B in its own fresh runtime.**

In [ ]:
# B1 -- generation deps. FRESH RUNTIME WITH A GPU.
#
# Checks run cheapest-first, so a wrong runtime fails in seconds rather than
# after several GB of wheels.
#
# Three bugs this cell had on 2026-09-02, all the same shape -- a check more
# confident than its evidence:
#
#   1. It blamed transformers for a torchvision ABI break. transformers' lazy
#      loader catches torchvision's RuntimeError and re-raises it as
#      "Could not import module 'LlamaModel'", and this cell believed the
#      re-raise. Its advice -- pin transformers==4.46.3 -- would have broken
#      the pin chatterbox actually declares while leaving the real problem in
#      place. transformers was already at exactly the declared 5.2.0.
#   2. It imported torch for the GPU check BEFORE the pip step, then probed
#      the stale in-memory modules afterwards, reporting torch 2.10.0 while
#      pip had just written 2.6.0 to disk. You cannot validate a pip change in
#      the kernel that pip changed.
#   3. It read "torchvision is not installed" as "torchvision is broken",
#      uninstalled nothing, and demanded a restart -- forever.
import subprocess, sys

# --- 0. GPU, before anything is downloaded ---------------------------------
# nvidia-smi, NOT `import torch`: importing torch here is precisely what made
# every check below it meaningless last time.
smi = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
if smi.returncode != 0 or not smi.stdout.strip():
    raise SystemExit(
        "No GPU attached. Runtime > Change runtime type > T4 GPU, then re-run.\n"
        "CPU generation runs at ~90s per clip; 120 cases would take hours, and "
        "nothing in the output would say so."
    )
print("GPU:", smi.stdout.strip())

# --- 1. chatterbox, and the transformers pin it declares -------------------
# `pip install chatterbox-tts` alone is not enough on Colab: pip does not
# downgrade transformers because chatterbox's pin reads as already satisfied.
# Read the pin out of chatterbox's OWN metadata rather than hardcoding a
# version to guess at.
get_ipython().system('pip install -q chatterbox-tts soundfile librosa')

import importlib.metadata as md

reqs = md.requires("chatterbox-tts") or []
tf_req = next((r for r in reqs if r.lower().startswith("transformers")), None)
print("chatterbox-tts declares:", tf_req or "(no transformers pin declared)")
print("currently installed transformers:", md.version("transformers"))
if tf_req:
    # deps untouched so this cannot cascade into torch.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", tf_req], check=False)
    print("-> transformers now:", md.version("transformers"))

# --- 2. stale-kernel guard, BEFORE any conclusion is drawn from an import ---
# Same in-memory-vs-on-disk check cell A1 uses for numpy. chatterbox pins
# torch==2.6.0, which downgrades Colab's torch on disk; anything this kernel
# already imported is now a different build from what is installed, and the
# first C extension to load says so in a way that names the wrong package
# ("numpy.dtype size changed", "operator torchvision::nms does not exist").
stale = []
for mod in ("torch", "numpy"):
    live = sys.modules.get(mod)
    if live is None:
        continue
    in_mem = getattr(live, "__version__", "")
    on_disk = md.version(mod)
    if in_mem.split("+")[0] != on_disk.split("+")[0]:
        stale.append(f"  {mod}: in memory {in_mem}, on disk {on_disk}")
if stale:
    raise SystemExit(
        "STOP -- restart the runtime.\n\n" + "\n".join(stale) + "\n\n"
        "pip replaced these underneath a kernel that had already imported them, "
        "so nothing checked after this point means anything.\n\n"
        "Runtime > RESTART SESSION (not 'delete runtime' -- pip state survives a "
        "restart, so torch is not re-downloaded), then re-run this cell."
    )

import torch
print(f"torch: {torch.__version__} | cuda: {torch.cuda.is_available()}")

# --- 3. torchvision: decide from METADATA, never by importing it -----------
# Colab's torchvision is ABI-built against the torch it shipped with. Once
# chatterbox moves torch, torchvision::nms never registers, transformers'
# image_utils.py imports torchvision at module scope, and every transformers
# model import detonates.
#
# The previous version of this check found that out by IMPORTING torchvision,
# which half-loads a broken C extension into the kernel -- so the only way out
# was uninstall + restart + re-run, and that is a loop you sit through for no
# reason. torchvision DECLARES the torch it was built for; comparing that to
# the installed torch answers the same question with no import and no restart.
tv_verdict = "absent"
try:
    tv_version = md.version("torchvision")
    pin = next((r for r in (md.requires("torchvision") or [])
                if r.replace(" ", "").lower().startswith("torch==")), None)
    wants = pin.split("==", 1)[1].strip() if pin else None
    have = md.version("torch").split("+")[0]
    if wants and wants.split("+")[0] != have:
        print(f"torchvision {tv_version} was built for torch {wants}, but torch {have} "
              f"is installed -- torchvision::nms will not register.")
        print("Removing it: nothing in section B is a vision model, and transformers "
              "falls back to its non-torchvision path.")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"],
                       check=False)
        tv_verdict = "removed (never imported, so no restart needed)"
    else:
        import torchvision                 # safe: metadata says it matches
        torch.ops.torchvision.nms          # and this proves the op registered
        tv_verdict = f"present and working ({tv_version})"
except md.PackageNotFoundError:
    tv_verdict = "absent -- expected and fine"
except Exception as exc:
    # Metadata said it should work and it still did not. That IS worth a
    # restart, because the import above has now half-loaded it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"],
                   check=False)
    raise SystemExit(
        f"torchvision passed its metadata check but failed to load "
        f"({type(exc).__name__}: {exc}). Removed it. Runtime > RESTART SESSION, "
        "then re-run this cell."
    )
print("torchvision:", tv_verdict)

# --- 4. prove the import chain before spending GPU minutes -----------------
try:
    from transformers.models.llama import LlamaModel      # noqa: F401
    from chatterbox.tts_turbo import ChatterboxTurboTTS   # noqa: F401
    print("imports OK")
except Exception as exc:
    raise SystemExit(
        f"import still failing: {type(exc).__name__}: {exc}\n\n"
        "torchvision and the kernel both passed their probes, so this is "
        "something else. Read the INNERMOST exception in the traceback above "
        "before pinning any version -- the outermost one has been wrong twice."
    )

print("\nReady. The starlette / google-adk conflicts pip printed above are "
      "Colab's own preinstalled packages -- nothing here touches them.")


In [ ]:
# B2 — generate. 60 per split gives every declared voice_scam combination
# a usable number of cases (currently 82 total across all three).
!python generate/generate_voice_attacks.py --n-per-split 60 --seed 42

In [ ]:
# B2b -- backfill BEFORE the eval, never after.
#
# evaluation_results.case_id is a FOREIGN KEY into attack_cases. If the cases
# just generated are not in that table, every insert batch the eval makes is
# rejected -- and each eval script wraps its persistence in `except Exception`
# and prints "skipped (non-fatal)" to stderr, which Colab does not surface.
# That is exactly how a document run reported 480 scored and 0 persisted.
#
# BOTH scripts are needed, and the omission of the second one WAS that bug:
#   backfill_attack_cases.py      -- data/generated/attacks/ only, i.e. the
#                                    four tabular families.
#   backfill_phase2_artifacts.py  -- document_fraud, voice_scam,
#                                    video_kyc_impersonation, phishing_scam.
#                                    It also uploads the media to Storage and
#                                    writes artifacts.audio_url / image_url /
#                                    video_url -- the URLs the evidence viewer
#                                    actually plays.
!python generate/run_all_generation.py --only backfill_attack_cases,backfill_phase2_artifacts


In [ ]:
# B3 — spoof-model bake-off over the identical case set.
!pip install -q transformers
import os, subprocess

MODELS = [
    "garystafford/wav2vec2-deepfake-voice-detector",   # incumbent
    "mo-thecreator/Deepfake-audio-detection",
    "Hemgg/Deepfake-audio-detection",
]
for model_id in MODELS:
    print("\n" + "=" * 70, f"\n{model_id}\n" + "=" * 70)
    env = {**os.environ, "VOICE_MODEL_ID": model_id}
    # A challenger that fails to load is a real answer about that checkpoint,
    # not a reason to stop the bake-off.
    subprocess.run(["python", "evaluation/eval_voice_spoof.py"], env=env)

In [ ]:
# B4 — compare
import json, pathlib
m = json.loads(pathlib.Path("defend/models/metrics.json").read_text())
rows = {k: v for k, v in m.items() if k.startswith("voice_spoof_detector")}
print(f"{'entry':52} {'recall':>8} {'prec':>8} {'FPR':>8} {'n':>6}")
for k, v in sorted(rows.items()):
    e = v.get("metrics", v)
    print(f"{k:52} {e.get('recall', 0):8.4f} {e.get('precision', 0):8.4f} "
          f"{e.get('false_positive_rate', 0):8.4f} {e.get('n_samples', 0):6}")

In [ ]:
# B5 -- CHECKPOINT: get section B's results out of this runtime.
#
# Run it after generation AND again after the eval. Idempotent: it pushes
# whatever exists right now, so an early run is never wasted.
#
# Every step runs even if an earlier one fails, and each reports its own
# status at the end. A step that fails quietly is exactly how ten
# evaluation_runs rows ended up marked "completed" with zero result rows.
import json, os, pathlib, subprocess, sys

REPO = pathlib.Path("/content/red_hat_vs_blue_hat_attack")
BACKEND = REPO / "backend"
BUNDLES = "voice_attacks,voice_bonafide,synthetic_customers"
FAMILY = "voice_scam"
METRIC_PREFIX = "voice_spoof_detector"
status = {}


def step(name, argv):
    print("\n" + "=" * 70 + f"\n{name}\n" + "=" * 70, flush=True)
    try:
        rc = subprocess.run(argv, cwd=BACKEND).returncode
    except Exception as exc:
        status[name] = f"FAILED ({type(exc).__name__}: {exc})"
        return False
    status[name] = "ok" if rc == 0 else f"FAILED (exit {rc})"
    return rc == 0


# --- 1. generated media + case JSONs -> Supabase Storage -------------------
step("1. storage_sync push", [sys.executable, "tools/storage_sync.py", "push", "--only", BUNDLES])

# --- 2. case JSONs -> attack_cases rows, AND media -> Storage --------------
# backfill_attack_cases.py alone is NOT enough and that omission was a real
# bug: it scans data/generated/attacks/ only, i.e. the four tabular families.
# The media families come from backfill_phase2_artifacts.py, which is also
# what uploads the audio/image/video and writes artifacts.*_url.
step("2. backfill cases + media URLs",
     [sys.executable, "generate/run_all_generation.py", "--only",
      "backfill_attack_cases,backfill_phase2_artifacts"])

# --- 3. metrics.json + EVALUATION_RESULTS.md -> Drive ----------------------
# No token, no network beyond Drive. This is the copy that survives if the
# git push below cannot run, so it comes FIRST.
print("\n" + "=" * 70 + "\n3. copy metrics + results log to Drive\n" + "=" * 70)
drive = pathlib.Path("/content/drive/MyDrive/fraudshield_eval_cache")
if not drive.exists():
    status["3. Drive copy"] = "SKIPPED (Drive not mounted -- run cell 3b)"
    print("Drive is not mounted. Run section 3b if you want a token-free backup.")
else:
    try:
        for rel in ["backend/defend/models/metrics.json", "docs/EVALUATION_RESULTS.md"]:
            src = REPO / rel
            if src.exists():
                dest = drive / f"{FAMILY}__{pathlib.Path(rel).name}"
                dest.write_bytes(src.read_bytes())
                print(f"  {rel} -> {dest}")
        status["3. Drive copy"] = "ok"
    except Exception as exc:
        status["3. Drive copy"] = f"FAILED ({type(exc).__name__}: {exc})"
        print(status["3. Drive copy"])

# --- 4. repo changes -> GitHub ---------------------------------------------
print("\n" + "=" * 70 + "\n4. git commit + push\n" + "=" * 70)
token = globals().get("GITHUB_TOKEN", "") or os.environ.get("GITHUB_TOKEN", "")
if not token:
    status["4. git push"] = "SKIPPED (no GITHUB_TOKEN -- step 3's Drive copy is your backup)"
    print("No GITHUB_TOKEN set in cell 1. Nothing pushed to GitHub.")
    print("The metrics are still in Drive (step 3) and the JSON below.")
else:
    subprocess.run(["git", "-C", str(REPO), "config", "user.email", "colab@fraudshield.local"])
    subprocess.run(["git", "-C", str(REPO), "config", "user.name", "FraudShield Colab runner"])
    subprocess.run(["git", "-C", str(REPO), "add",
                    "backend/defend/models/metrics.json", "docs/EVALUATION_RESULTS.md"])
    c = subprocess.run(["git", "-C", str(REPO), "commit", "-m",
                        f"colab: section B evidence-gate results ({FAMILY})"])
    if c.returncode != 0:
        print("nothing new to commit")
    rc = subprocess.run(["git", "-C", str(REPO), "push", "origin", "HEAD:main"]).returncode
    status["4. git push"] = "ok" if rc == 0 else f"FAILED (exit {rc})"

# --- 5. what actually landed ----------------------------------------------
print("\n" + "=" * 70 + "\n5. verify against Supabase (not against this script's own hopes)\n" + "=" * 70)
try:
    sys.path.insert(0, str(BACKEND))
    from db.supabase_client import get_service_client
    sb = get_service_client()
    cases = sb.table("attack_cases").select("case_id", count="exact").eq("attack_family", FAMILY).execute()
    res = sb.table("evaluation_results").select("id", count="exact").like("case_id", f"{FAMILY}%").execute()
    print(f"  attack_cases         ({FAMILY}): {cases.count}")
    print(f"  evaluation_results   ({FAMILY}): {res.count}")
    status["5. verify"] = "ok"
except Exception as exc:
    status["5. verify"] = f"FAILED ({type(exc).__name__}: {exc})"
    print(status["5. verify"])

# --- 6. the metrics entries, to paste back --------------------------------
print("\n" + "=" * 70 + "\n6. metrics.json entries for voice_spoof_detector* -- paste these back\n" + "=" * 70)
try:
    m = json.loads((BACKEND / "defend/models/metrics.json").read_text())
    entries = {k: v for k, v in m.items() if k.startswith(METRIC_PREFIX)}
    print(json.dumps(entries, indent=2) if entries else f"(no {METRIC_PREFIX}* entry yet)")
except Exception as exc:
    print(f"could not read metrics.json: {type(exc).__name__}: {exc}")

print("\n" + "=" * 70 + "\nCHECKPOINT SUMMARY\n" + "=" * 70)
for k, v in status.items():
    print(f"  {k:28} {v}")
if any(str(v).startswith("FAILED") for v in status.values()):
    print("\nAt least one step FAILED -- do not treat this section as checkpointed.")
else:
    print("\nAll steps ok. This runtime can die without losing section B.")


---
## C · Video-KYC  (facenet-pytorch, GPU)

`video_kyc_detector` currently reports precision/recall/F1 = 1.000 on
**n = 12**. That is a wiring check, not a result, and it is the single most
attackable number on the site. This section generates a real set and re-scores.

facenet-pytorch pins `torch<2.3` while `transformers` needs `torch>=2.5` — they
cannot share an environment (that is why the Railway image excludes video-KYC).
**Run section C in its own fresh runtime.**

In [ ]:
# C1 — install. FRESH RUNTIME, please (Runtime > Disconnect and delete runtime).
#
# This is the one section that needs numpy<2: facenet-pytorch 2.6.0 requires
# numpy<2.0.0, while Colab's preinstalled stack is numpy 2.x. Expect pip to
# report conflicts against jax/cupy/shap and friends -- those are Colab's own
# packages, not ours, and nothing in this section touches them.
!pip install -q "numpy<2" facenet-pytorch opencv-python-headless

import numpy, torch
print("numpy:", numpy.__version__, "| torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
assert numpy.__version__.startswith("1."), (
    "facenet-pytorch needs numpy<2. Runtime > Restart session, then re-run this cell "
    "WITHOUT re-running section A in the same runtime."
)
from facenet_pytorch import MTCNN, InceptionResnetV1
print("facenet ready")

In [ ]:
# C2 -- pull the media, then generate. NO eval in this cell.
# Splitting generation from evaluation so a failure in one is not reported as
# the other, and so the backfill can sit between them (see C2b).
!python tools/storage_sync.py pull --only video_kyc_attacks,video_kyc_bonafide,video_kyc_reference,synthetic_customers
!python generate/generate_video_kyc_attacks.py --n-per-split 60 --seed 42


In [ ]:
# C2b -- backfill BEFORE the eval. Same foreign-key reason as B2b.
#
# This family had NO persistence path at all until 2026-09-02: grep video_kyc
# across db/ and tools/ returned only sync_model_registry.py, and attack_cases
# held zero rows for it while metrics.json reported video_kyc_detector at
# 1.000 on n=12. So C's eval would have written metrics and persisted nothing.
# backfill_phase2_artifacts.py now uploads each video AND the reference photo
# it is compared against, and writes both URLs into artifacts.
!python generate/run_all_generation.py --only backfill_attack_cases,backfill_phase2_artifacts


In [ ]:
# C2c -- score. Only now, with the rows in place, can persistence succeed.
!python evaluation/eval_video_kyc.py


In [ ]:
# C3 -- CHECKPOINT: get section C's results out of this runtime.
#
# Run it after generation AND again after the eval. Idempotent: it pushes
# whatever exists right now, so an early run is never wasted.
#
# Every step runs even if an earlier one fails, and each reports its own
# status at the end. A step that fails quietly is exactly how ten
# evaluation_runs rows ended up marked "completed" with zero result rows.
import json, os, pathlib, subprocess, sys

REPO = pathlib.Path("/content/red_hat_vs_blue_hat_attack")
BACKEND = REPO / "backend"
BUNDLES = "video_kyc_attacks,video_kyc_bonafide,video_kyc_reference,synthetic_customers"
FAMILY = "video_kyc_impersonation"
METRIC_PREFIX = "video_kyc_detector"
status = {}


def step(name, argv):
    print("\n" + "=" * 70 + f"\n{name}\n" + "=" * 70, flush=True)
    try:
        rc = subprocess.run(argv, cwd=BACKEND).returncode
    except Exception as exc:
        status[name] = f"FAILED ({type(exc).__name__}: {exc})"
        return False
    status[name] = "ok" if rc == 0 else f"FAILED (exit {rc})"
    return rc == 0


# --- 1. generated media + case JSONs -> Supabase Storage -------------------
step("1. storage_sync push", [sys.executable, "tools/storage_sync.py", "push", "--only", BUNDLES])

# --- 2. case JSONs -> attack_cases rows, AND media -> Storage --------------
# backfill_attack_cases.py alone is NOT enough and that omission was a real
# bug: it scans data/generated/attacks/ only, i.e. the four tabular families.
# The media families come from backfill_phase2_artifacts.py, which is also
# what uploads the audio/image/video and writes artifacts.*_url.
step("2. backfill cases + media URLs",
     [sys.executable, "generate/run_all_generation.py", "--only",
      "backfill_attack_cases,backfill_phase2_artifacts"])

# --- 3. metrics.json + EVALUATION_RESULTS.md -> Drive ----------------------
# No token, no network beyond Drive. This is the copy that survives if the
# git push below cannot run, so it comes FIRST.
print("\n" + "=" * 70 + "\n3. copy metrics + results log to Drive\n" + "=" * 70)
drive = pathlib.Path("/content/drive/MyDrive/fraudshield_eval_cache")
if not drive.exists():
    status["3. Drive copy"] = "SKIPPED (Drive not mounted -- run cell 3b)"
    print("Drive is not mounted. Run section 3b if you want a token-free backup.")
else:
    try:
        for rel in ["backend/defend/models/metrics.json", "docs/EVALUATION_RESULTS.md"]:
            src = REPO / rel
            if src.exists():
                dest = drive / f"{FAMILY}__{pathlib.Path(rel).name}"
                dest.write_bytes(src.read_bytes())
                print(f"  {rel} -> {dest}")
        status["3. Drive copy"] = "ok"
    except Exception as exc:
        status["3. Drive copy"] = f"FAILED ({type(exc).__name__}: {exc})"
        print(status["3. Drive copy"])

# --- 4. repo changes -> GitHub ---------------------------------------------
print("\n" + "=" * 70 + "\n4. git commit + push\n" + "=" * 70)
token = globals().get("GITHUB_TOKEN", "") or os.environ.get("GITHUB_TOKEN", "")
if not token:
    status["4. git push"] = "SKIPPED (no GITHUB_TOKEN -- step 3's Drive copy is your backup)"
    print("No GITHUB_TOKEN set in cell 1. Nothing pushed to GitHub.")
    print("The metrics are still in Drive (step 3) and the JSON below.")
else:
    subprocess.run(["git", "-C", str(REPO), "config", "user.email", "colab@fraudshield.local"])
    subprocess.run(["git", "-C", str(REPO), "config", "user.name", "FraudShield Colab runner"])
    subprocess.run(["git", "-C", str(REPO), "add",
                    "backend/defend/models/metrics.json", "docs/EVALUATION_RESULTS.md"])
    c = subprocess.run(["git", "-C", str(REPO), "commit", "-m",
                        f"colab: section C evidence-gate results ({FAMILY})"])
    if c.returncode != 0:
        print("nothing new to commit")
    rc = subprocess.run(["git", "-C", str(REPO), "push", "origin", "HEAD:main"]).returncode
    status["4. git push"] = "ok" if rc == 0 else f"FAILED (exit {rc})"

# --- 5. what actually landed ----------------------------------------------
print("\n" + "=" * 70 + "\n5. verify against Supabase (not against this script's own hopes)\n" + "=" * 70)
try:
    sys.path.insert(0, str(BACKEND))
    from db.supabase_client import get_service_client
    sb = get_service_client()
    cases = sb.table("attack_cases").select("case_id", count="exact").eq("attack_family", FAMILY).execute()
    res = sb.table("evaluation_results").select("id", count="exact").like("case_id", f"{FAMILY}%").execute()
    print(f"  attack_cases         ({FAMILY}): {cases.count}")
    print(f"  evaluation_results   ({FAMILY}): {res.count}")
    status["5. verify"] = "ok"
except Exception as exc:
    status["5. verify"] = f"FAILED ({type(exc).__name__}: {exc})"
    print(status["5. verify"])

# --- 6. the metrics entries, to paste back --------------------------------
print("\n" + "=" * 70 + "\n6. metrics.json entries for video_kyc_detector* -- paste these back\n" + "=" * 70)
try:
    m = json.loads((BACKEND / "defend/models/metrics.json").read_text())
    entries = {k: v for k, v in m.items() if k.startswith(METRIC_PREFIX)}
    print(json.dumps(entries, indent=2) if entries else f"(no {METRIC_PREFIX}* entry yet)")
except Exception as exc:
    print(f"could not read metrics.json: {type(exc).__name__}: {exc}")

print("\n" + "=" * 70 + "\nCHECKPOINT SUMMARY\n" + "=" * 70)
for k, v in status.items():
    print(f"  {k:28} {v}")
if any(str(v).startswith("FAILED") for v in status.values()):
    print("\nAt least one step FAILED -- do not treat this section as checkpointed.")
else:
    print("\nAll steps ok. This runtime can die without losing section C.")


---
## D · GNN — mule-network round 6

Round 5 stands at IBM AML ROC-AUC **0.7532**, recall **0.0746**, F1 **0.0258**.

Read `notebooks/train_gnn_mule_network.ipynb` for the round-5 training loop; this
cell is the harness around it. Two things worth being precise about before
chasing a number:

1. **Recall alone is trivially purchasable.** With ROC-AUC 0.75 you can hit 60%
   recall today by lowering the decision threshold — and precision, already
   0.0156, collapses further. A 60% recall figure obtained that way is not a
   result, and reporting it as one would be exactly the kind of number this
   project exists to avoid.
2. **The real lever is ROC-AUC**, i.e. ranking quality: class-imbalance handling
   in the loss (`pos_weight`), longer training, and richer edge features. Recall
   at a *stated* precision is the honest headline.

So this section records the full precision/recall curve and reports the operating
points, rather than a single recall number picked to look good.

In [ ]:
# D1 — install
!pip install -q torch torch_geometric
import torch; print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# D2 — train. Open notebooks/train_gnn_mule_network.ipynb and run its cells
# here, or import its training function once it has been extracted into a
# module. The round-5 loop is the starting point; round 6 changes to try, in
# the order most likely to move ROC-AUC:
#
#   a) pos_weight in BCEWithLogitsLoss set to (neg/pos) on the training split.
#      IBM AML is ~0.1% positive; an unweighted loss learns to predict "no".
#   b) more epochs with early stopping on validation ROC-AUC, not on loss.
#   c) edge features: log-amount, time-delta to the account's previous edge,
#      in/out degree ratio at both endpoints, and the port-numbering features
#      round 5 added.
#
# Save the checkpoint to backend/defend/models/gnn.pt when done.
print("open notebooks/train_gnn_mule_network.ipynb and run its training cells")

In [ ]:
# D3 — evaluate the new checkpoint, and report operating points honestly
!python evaluation/eval_gnn.py

import json, pathlib
m = json.loads(pathlib.Path("defend/models/metrics.json").read_text())
for k in sorted(m):
    if k.startswith("gnn"):
        e = m[k].get("metrics", m[k])
        print(f"{k:48} roc_auc={e.get('roc_auc')} recall={e.get('recall')} precision={e.get('precision')}")

---
## E · Push everything back

Each eval script already wrote its `evaluation_runs` / `evaluation_results` rows
and updated `metrics.json` as it ran. This section pushes the two things that
live outside Supabase's tables: the regenerated dataset bundles, and the
`metrics.json` / `EVALUATION_RESULTS.md` changes in the repo.

In [ ]:
# E1 — dataset bundles back to Storage
!python tools/storage_sync.py push
!python db/sync_model_registry.py

In [ ]:
# E2 — repo changes back to GitHub.
# Needs GITHUB_TOKEN set in cell 1 (a token with repo write scope).
!git -C {REPO_DIR} config user.email "colab@fraudshield.local"
!git -C {REPO_DIR} config user.name "FraudShield Colab runner"
!git -C {REPO_DIR} add backend/defend/models/metrics.json backend/defend/models/gnn.pt docs/EVALUATION_RESULTS.md
!git -C {REPO_DIR} commit -m "colab: real evidence-gate results from the master runner" || echo "nothing to commit"
!git -C {REPO_DIR} push origin main

In [ ]:
# E3 — verify what actually landed in Supabase
import os
from supabase import create_client
sb = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_ROLE_KEY"])
for family in ["document_fraud", "voice_scam", "video_kyc"]:
    r = sb.table("evaluation_results").select("id", count="exact").like("case_id", f"{family}%").execute()
    print(f"{family:16} {r.count:>6} scored results in evaluation_results")